# Mining Duplicate Questions in Stack Overflow
**Libraries used:** `pandas`, `matplotlib` only

## Step 0 - Import Libraries and File Names

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

POSTLINKS_FILE = r"D:\\ict\\sem5\\ML\\LHC\\LHC1\\Arjun_Assignment_data-20220427T165022Z-002\\Arjun_Assignment_data\dataset-20210607T020316Z-001\\dataset\\postLinks.csv"
POSTS_FILE = r"D:\\ict\sem5\\ML\\LHC\\LHC1\\Arjun_Assignment_data-20220427T165022Z-002\\Arjun_Assignment_data\\dataset-20210607T020316Z-001\dataset\\posts_short.csv"
USERS_FILE = r"D:\\ict\\sem5\\ML\\LHC\\LHC1\\Arjun_Assignment_data-20220427T165022Z-002\\Arjun_Assignment_data\\dataset-20210607T020316Z-002\\dataset\\users.csv"
HISTORY_FILE = r"D:\\ict\\sem5\\ML\\LHC\\LHC1\\Arjun_Assignment_data-20220427T165022Z-002\\Arjun_Assignment_data\\dataset-20210607T020316Z-003\dataset\\post_history.csv"

## Step 1 - Load the Data
- `post_type_id == 1` means the post is a **question**
- All questions are kept, even the ones with no tags

In [ ]:
postlinks = pd.read_csv(POSTLINKS_FILE, usecols=["post_id", "related_post_id", "link_type_id"])
posts = pd.read_csv(POSTS_FILE, usecols=["id", "post_type_id", "creation_date", "owner_user_id", "tags"])
users = pd.read_csv(USERS_FILE, usecols=["id", "reputation"])

users = users.rename(columns={"id": "owner_user_id"})

questions = posts[posts["post_type_id"] == 1].copy()
questions["creation_date"] = pd.to_datetime(questions["creation_date"], errors="coerce")

print("Total questions:", len(questions))

## Step 2 - Find Duplicate Questions
`link_type_id == 3` means the post is a **duplicate** of another post.

In [ ]:
duplicate_ids = postlinks[postlinks["link_type_id"] == 3]["post_id"]

questions["is_duplicate"] = questions["id"].isin(duplicate_ids)

print("Duplicate questions:", questions["is_duplicate"].sum())
print("Normal questions   :", (~questions["is_duplicate"]).sum())

## Step 3 - Split Tags and Count Them
Tags look like `<python><pandas>`. `findall` picks every word between `<` and `>` and gives a list `["python", "pandas"]`. A question with no tags gets an empty list.
`explode()` puts each tag on its own row, then `value_counts()` counts them.

In [ ]:
questions["tag_list"] = questions["tags"].fillna("").str.findall("<([^>]+)>")

all_tags = questions["tag_list"].explode()
tag_counts = all_tags.value_counts()

print("Total tag uses:", all_tags.count())
print("Unique tags   :", len(tag_counts))

---
## Q1 - Number of Tags Per Question
Count how many tags each question has.

In [ ]:
questions["num_tags"] = questions["tag_list"].apply(len)
tags_per_question = questions["num_tags"].value_counts().sort_index()

print(tags_per_question)
print("Average tags per question:", round(questions["num_tags"].mean(), 2))

plt.figure(figsize=(8, 5))
plt.bar(tags_per_question.index, tags_per_question.values, color="steelblue")
plt.title("Q1: Number of Tags Per Question")
plt.xlabel("Number of Tags")
plt.ylabel("Number of Questions")
plt.show()

---
## Q2 - Total Number of Unique Tags

In [ ]:
print("Total unique tags:", len(tag_counts))

---
## Q3 - Top 25 Most Frequent Tags

In [ ]:
top25 = tag_counts.head(25)
print(top25)

plt.figure(figsize=(12, 5))
plt.bar(top25.index, top25.values, color="darkorange")
plt.title("Q3: Top 25 Most Frequent Tags")
plt.xlabel("Tag")
plt.ylabel("Number of Questions")
plt.xticks(rotation=60, ha="right")
plt.show()

---
## Q4 - Distribution of Top 500 Tags
We plot tag frequency against rank, once on a normal scale and once on a log-log scale. On a log-log scale a power law looks like a straight line going down.

**Answer:** It is a **power-law (long-tail) distribution**. A few tags are used a huge number of times and most tags are used rarely.

In [ ]:
top500 = tag_counts.head(500).values
ranks = list(range(1, 501))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(ranks, top500, color="purple")
plt.title("Normal Scale")
plt.xlabel("Rank")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.plot(ranks, top500, color="purple")
plt.xscale("log")
plt.yscale("log")
plt.title("Log-Log Scale")
plt.xlabel("Rank (log)")
plt.ylabel("Frequency (log)")

plt.show()

---
## Q5 - Ratio of Duplicate Questions Per Month (2008 to 2018)
Average of `is_duplicate` (True = 1, False = 0) in each month gives the ratio of duplicates.

In [ ]:
years = questions["creation_date"].dt.year
selected = questions[(years >= 2008) & (years <= 2018)]

month = selected["creation_date"].dt.to_period("M")
monthly_ratio = selected.groupby(month)["is_duplicate"].mean() * 100

print("Highest ratio:", round(monthly_ratio.max(), 2), "%")
print("Average ratio:", round(monthly_ratio.mean(), 2), "%")

plt.figure(figsize=(12, 5))
plt.plot(monthly_ratio.index.to_timestamp(), monthly_ratio.values, color="crimson")
plt.title("Q5: Ratio of Duplicate Questions Per Month")
plt.xlabel("Month")
plt.ylabel("Duplicate %")
plt.show()

---
## Q6 - Top 20 Tags With Highest % of Duplicate Questions
For each tag: `duplicate questions with the tag / all questions with the tag x 100`.
Only tags used in at least `MIN_QUESTIONS` questions are kept (50 here), because a tag used in 1 question that is a duplicate would show 100%. Change `MIN_QUESTIONS` to 1 to see the result without any filter.

In [ ]:
tag_table = questions[["tag_list", "is_duplicate"]].explode("tag_list")

stats = tag_table.groupby("tag_list")["is_duplicate"].agg(["count", "mean"])
MIN_QUESTIONS = 50
stats = stats[stats["count"] >= MIN_QUESTIONS]
stats["dup_percent"] = stats["mean"] * 100

top20 = stats.sort_values("dup_percent", ascending=False).head(20)
print(top20[["count", "dup_percent"]])

plt.figure(figsize=(12, 5))
plt.bar(top20.index, top20["dup_percent"], color="teal")
plt.title("Q6: Top 20 Tags by Duplicate %")
plt.xlabel("Tag")
plt.ylabel("Duplicate %")
plt.xticks(rotation=60, ha="right")
plt.show()

---
## Q7 - Time Taken to Close Duplicate Questions
- `post_history.csv` has the history of every post
- `ph_type_id == 10` means the post was **closed**
- A post can be closed more than once, so we take the **first** close date
- Time to close = close date - date the question was asked

In [ ]:
history = pd.read_csv(HISTORY_FILE, usecols=["post_id", "ph_type_id", "creation_date"])

closed = history[history["ph_type_id"] == 10].copy()
closed["close_date"] = pd.to_datetime(closed["creation_date"], errors="coerce")
closed = closed.groupby("post_id")["close_date"].min().reset_index()

dups = questions[questions["is_duplicate"]][["id", "creation_date"]]
dups = dups.rename(columns={"id": "post_id", "creation_date": "asked_date"})

merged = dups.merge(closed, on="post_id")
merged["days_to_close"] = (merged["close_date"] - merged["asked_date"]).dt.total_seconds() / 86400
merged = merged[merged["days_to_close"] >= 0]

print("Average days to close:", round(merged["days_to_close"].mean(), 1))
print("Median days to close :", round(merged["days_to_close"].median(), 1))

bins = [0, 1, 7, 30, 180, 365, float("inf")]
labels = ["< 1 day", "1-7 days", "1-4 weeks", "1-6 months", "6-12 months", "> 1 year"]
merged["time_group"] = pd.cut(merged["days_to_close"], bins=bins, labels=labels, right=False)
time_counts = merged["time_group"].value_counts().sort_index()
print(time_counts)

plt.figure(figsize=(9, 5))
plt.bar(time_counts.index, time_counts.values, color="salmon")
plt.title("Q7: Time Taken to Close Duplicate Questions")
plt.xlabel("Time to Close")
plt.ylabel("Number of Questions")
plt.xticks(rotation=30, ha="right")
plt.show()

---
## Q8 - Reputation of Users Whose Questions Were Closed as Duplicates
Goal: check whether only new (low reputation) users ask duplicate questions.

We take duplicate questions that were closed (`closed` from Q7), find their owners, count each user **once**, and group users by reputation.

**Answer:** If users in the high reputation ranges also have bars, then duplicate questions are **not** asked only by low reputation users.

In [ ]:
dup_closed = questions[questions["is_duplicate"] & questions["id"].isin(closed["post_id"])]
dup_users = dup_closed.merge(users, on="owner_user_id")
dup_users = dup_users.drop_duplicates("owner_user_id")

bins = [0, 10, 50, 100, 500, 1000, 5000, 10000, 50000, float("inf")]
labels = ["0-9", "10-49", "50-99", "100-499", "500-999", "1K-5K", "5K-10K", "10K-50K", "50K+"]
dup_users["rep_group"] = pd.cut(dup_users["reputation"], bins=bins, labels=labels, right=False)
rep_counts = dup_users["rep_group"].value_counts().sort_index()
print(rep_counts)

low_percent = (dup_users["reputation"] < 100).mean() * 100
print("Users with reputation below 100:", round(low_percent, 1), "%")

plt.figure(figsize=(10, 5))
plt.bar(rep_counts.index, rep_counts.values)
plt.title("Q8: Reputation of Users With Closed Duplicate Questions")
plt.xlabel("Reputation Range")
plt.ylabel("Number of Users")
plt.xticks(rotation=30, ha="right")
plt.show()

---
## Q9 - % of Questions Covered by Top 500 Tags
A question is **covered** if at least one of its tags is in the top 500 tags.

The total used for the percentage is **all** questions, including the ones with no tags.

The last line also shows a stricter meaning: questions where **all** tags are in the top 500.

In [ ]:
tag_table2 = questions[["id", "tag_list"]].explode("tag_list").dropna()

top500_names = tag_counts.head(500).index
covered_500 = tag_table2[tag_table2["tag_list"].isin(top500_names)]["id"].nunique()
percent_500 = covered_500 / len(questions) * 100

print("Total questions     :", len(questions))
print("Covered by top 500  :", covered_500)
print("Coverage            :", round(percent_500, 2), "%")

not_covered_ids = tag_table2[~tag_table2["tag_list"].isin(top500_names)]["id"].nunique()
all_covered_500 = tag_table2["id"].nunique() - not_covered_ids
print("All tags in top 500 :", round(all_covered_500 / len(questions) * 100, 2), "%")

---
## Q10 - % of Questions Covered by Top 5000 Tags
Same as Q9 but with 5000 tags, then we compare both.

In [ ]:
top5000_names = tag_counts.head(5000).index
covered_5000 = tag_table2[tag_table2["tag_list"].isin(top5000_names)]["id"].nunique()
percent_5000 = covered_5000 / len(questions) * 100

print("Covered by top 5000 :", covered_5000)
print("Coverage            :", round(percent_5000, 2), "%")
print("Extra coverage      :", round(percent_5000 - percent_500, 2), "%")

not_covered_ids = tag_table2[~tag_table2["tag_list"].isin(top5000_names)]["id"].nunique()
all_covered_5000 = tag_table2["id"].nunique() - not_covered_ids
print("All tags in top 5000:", round(all_covered_5000 / len(questions) * 100, 2), "%")

plt.figure(figsize=(6, 5))
plt.bar(["Top 500 Tags", "Top 5000 Tags"], [percent_500, percent_5000])
plt.title("Q9 and Q10: Questions Covered by Top Tags")
plt.ylabel("Coverage %")
plt.ylim(0, 105)
plt.show()

---
## Quick Viva Notes

| Term | Simple meaning |
|------|----------------|
| `link_type_id == 3` | The post is a duplicate of another post |
| `ph_type_id == 10` | The post was closed |
| `post_type_id == 1` | The post is a question |
| `str.findall("<([^>]+)>")` | Picks every tag between `<` and `>` and makes a list |
| `explode()` | Gives each tag its own row |
| `value_counts()` | Counts how many times each value appears |
| `groupby().mean()` | Average per group. On True/False data this gives the ratio |
| `pd.cut()` | Puts numbers into groups (bins) like 0-1 day, 1-7 days |
| `isin()` | Checks if a value is in a given list |
| Power law | Few items are very common, most items are rare. Straight line on a log-log plot |